# Baseline 3: Enhanced RAG System - Visualizing Data Flow

## 1. Data Flow Diagram
To understand how the Enhanced RAG system operates, let's track the data transformations step-by-step:
1. **Original Query** -> Extract fiscal year & apply hard filtering to the document list (`active_keys`).
2. **Original Query** -> Apply synonym expansion -> `expanded_query`.
3. **Parallel Retrieval**:
   - **BM25**: Search using `expanded_query` limited within the `active_keys` -> Returns BM25 ranking.
   - **Dense HNSW**: Search using `original_query` limited within the `active_keys` -> Returns HNSW ranking.
4. **RRF (Rank Fusion)**: Merge the two ranked lists based on their rank positions -> Returns the merged `candidates` list.
5. **Cross-Encoder**: Rerank the `candidates` -> Returns the final optimal results.

## 2. Initialize Data & Import Libraries

In [1]:
import numpy as np
import pandas as pd
import re
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

corpus = {
    "doc1": "Apple Net sales in fiscal year 2023 were $383,285 million.",
    "doc2": "Apple payments for acquisition of property plant and equipment were $10,959 million.",
    "doc3": "Amazon net sales were $574,785 million in fiscal year 2023.",
    "doc4": "Amazon purchases of property and equipment in fiscal year 2023 were $52,729 million.",
    "doc5": "NVIDIA net income was $29,760 million in fiscal year 2024."
}

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s$]', '', text)
    return text.split()

tokenized_docs = {k: tokenize(v) for k, v in corpus.items()}
vocabulary = sorted(list(set(word for doc in tokenized_docs.values() for word in doc)))

print(f"[Log] Initialization successful. Vocabulary size: {len(vocabulary)} words.")

c:\Users\huynh\Desktop\NLP-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Log] Initialization successful. Vocabulary size: 28 words.


## 3. Load Models

In [2]:
print("[Log] Loading Bi-Encoder (BGE) and Cross-Encoder (MiniLM) models...")
bi_encoder = SentenceTransformer("BAAI/bge-small-en-v1.5")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("[SUCCESS] Models loaded successfully!")

[Log] Loading Bi-Encoder (BGE) and Cross-Encoder (MiniLM) models...
[SUCCESS] Models loaded successfully!


## 4. Step-by-Step Walkthrough Demonstration
We run a test with the query: **"Amazon capital expenditures 2023"**

In [3]:
query = "Amazon capital expenditures 2023"
print(f"ORIGINAL INPUT: '{query}'")

ORIGINAL INPUT: 'Amazon capital expenditures 2023'


### Step 4.1: Year Routing
**Goal:** Extract the year from the query and apply a hard filter to keep only documents from that year.

In [4]:
# Extract year
match = re.search(r'\b(20\d{2})\b', query)
target_year = int(match.group(1)) if match else None
print(f"[Process] Extracted year: {target_year}")

# Filter active documents list (active_keys)
active_keys = [k for k, text in corpus.items() if not target_year or str(target_year) in text]

print(f"\nOUTPUT STEP 4.1:")
print(f"  - target_year = {target_year}")
print(f"  - active_keys = {active_keys}")
print("  (doc5 from year 2024 has been removed)")

[Process] Extracted year: 2023

OUTPUT STEP 4.1:
  - target_year = 2023
  - active_keys = ['doc1', 'doc3', 'doc4']
  (doc5 from year 2024 has been removed)


### Step 4.2: Query Expansion
**Goal:** Expand the query with financial synonyms to assist the keyword-based search engine.

In [5]:
SYNONYMS_DICT = {
    "capital expenditures": ["purchases of property and equipment", "capex"],
    "net sales": ["revenue", "sales"],
    "net income": ["profit", "net earnings"]
}

expanded_query = query
query_lower = query.lower()
for key_phrase, synonyms in SYNONYMS_DICT.items():
    if key_phrase in query_lower:
        expanded_query = query + " " + " ".join(synonyms)
        print(f"[Process] Phrase matched '{key_phrase}' -> Added: {synonyms}")
        break

print(f"\nOUTPUT STEP 4.2:")
print(f"  - expanded_query = '{expanded_query}'")

[Process] Phrase matched 'capital expenditures' -> Added: ['purchases of property and equipment', 'capex']

OUTPUT STEP 4.2:
  - expanded_query = 'Amazon capital expenditures 2023 purchases of property and equipment capex'


### Step 4.3: Parallel Retrieval
#### 1. BM25 Ranking (on `expanded_query` & `active_keys`)
Print the detailed term-level scoring matrix.

In [6]:
# Calculate static BM25 IDF
bm25_idf = {}
for word in vocabulary:
    n_qi = sum(1 for tokens in tokenized_docs.values() if word in tokens)
    bm25_idf[word] = np.log((len(corpus) - n_qi + 0.5) / (n_qi + 0.5) + 1.0)

doc_lengths = {k: len(v) for k, v in tokenized_docs.items()}
avgdl = sum(doc_lengths.values()) / len(doc_lengths)

k1, b = 1.2, 0.75
q_tokens = tokenize(expanded_query)
bm25_scores = {}

print("=== DETAILED WEIGHT MATRIX PER KEYWORD (BM25) ===")
for k in active_keys:
    tokens = tokenized_docs[k]
    doc_len = len(tokens)
    score = 0.0
    print(f"\n* Document {k} (length {doc_len}):")
    for word in q_tokens:
        if word in vocabulary:
            tf = tokens.count(word)
            idf = bm25_idf[word]
            if tf > 0:
                term_score = idf * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * (doc_len / avgdl)))
                score += term_score
                print(f"  - Word '{word}': tf={tf}, idf={idf:.4f} -> Score = {term_score:.4f}")
    bm25_scores[k] = score
    print(f"  => Total BM25 score for {k} = {score:.4f}")

bm25_ranked = [doc for doc, sc in sorted(bm25_scores.items(), key=lambda x: x[1], reverse=True) if sc > 0]
print(f"\nOUTPUT BM25 RANKING: {bm25_ranked}")

=== DETAILED WEIGHT MATRIX PER KEYWORD (BM25) ===

* Document doc1 (length 10):
  - Word '2023': tf=1, idf=0.5390 -> Score = 0.5598
  => Total BM25 score for doc1 = 0.5598

* Document doc3 (length 10):
  - Word 'amazon': tf=1, idf=0.8755 -> Score = 0.9093
  - Word '2023': tf=1, idf=0.5390 -> Score = 0.5598
  => Total BM25 score for doc3 = 1.4691

* Document doc4 (length 13):
  - Word 'amazon': tf=1, idf=0.8755 -> Score = 0.8149
  - Word '2023': tf=1, idf=0.5390 -> Score = 0.5017
  - Word 'purchases': tf=1, idf=1.3863 -> Score = 1.2903
  - Word 'of': tf=1, idf=0.8755 -> Score = 0.8149
  - Word 'property': tf=1, idf=0.8755 -> Score = 0.8149
  - Word 'and': tf=1, idf=0.8755 -> Score = 0.8149
  - Word 'equipment': tf=1, idf=0.8755 -> Score = 0.8149
  => Total BM25 score for doc4 = 5.8663

OUTPUT BM25 RANKING: ['doc4', 'doc3', 'doc1']


#### 2. Dense HNSW Ranking (on original `query` & `active_keys`)
Print the detailed Cosine Similarity calculation.

In [7]:
active_texts = [corpus[k] for k in active_keys]
active_embeddings = bi_encoder.encode(active_texts, normalize_embeddings=True)
query_vector = bi_encoder.encode([query], normalize_embeddings=True)[0]

hnsw_scores = {}
print("=== DENSE COSINE SIMILARITY MATRIX ===")
for i, k in enumerate(active_keys):
    doc_vector = active_embeddings[i]
    cosine_sim = np.dot(query_vector, doc_vector)
    hnsw_scores[k] = cosine_sim
    print(f"  - {k} vs Query: Cosine = {cosine_sim:.4f} (First 3 dimensions: q={query_vector[:3].round(3)}, d={doc_vector[:3].round(3)})")

hnsw_ranked = [doc for doc, sc in sorted(hnsw_scores.items(), key=lambda x: x[1], reverse=True)]
print(f"\nOUTPUT HNSW RANKING: {hnsw_ranked}")

=== DENSE COSINE SIMILARITY MATRIX ===
  - doc1 vs Query: Cosine = 0.6338 (First 3 dimensions: q=[-0.003 -0.073  0.04 ], d=[ 0.033 -0.033 -0.033])
  - doc3 vs Query: Cosine = 0.7555 (First 3 dimensions: q=[-0.003 -0.073  0.04 ], d=[ 0.009 -0.062  0.028])
  - doc4 vs Query: Cosine = 0.7641 (First 3 dimensions: q=[-0.003 -0.073  0.04 ], d=[ 0.018 -0.011 -0.031])

OUTPUT HNSW RANKING: ['doc4', 'doc3', 'doc1']


### Step 4.4: Rank Fusion using RRF (Reciprocal Rank Fusion)
*   **Input:** `bm25_ranked` and `hnsw_ranked` (outputs from Step 4.3).
*   **Action:** Compute RRF scores based on rank positions to merge results.
*   **Output:** Detailed RRF score table.

In [8]:
k_constant = 60
rrf_scores = {}
doc_track = {}

for rank, doc_name in enumerate(bm25_ranked):
    rank_1based = rank + 1
    score = 1.0 / (k_constant + rank_1based)
    rrf_scores[doc_name] = rrf_scores.get(doc_name, 0.0) + score
    doc_track[doc_name] = {'bm25_rank': rank_1based, 'bm25_score': score, 'hnsw_rank': 'N/A', 'hnsw_score': 0.0}

for rank, doc_name in enumerate(hnsw_ranked):
    rank_1based = rank + 1
    score = 1.0 / (k_constant + rank_1based)
    rrf_scores[doc_name] = rrf_scores.get(doc_name, 0.0) + score
    if doc_name in doc_track:
        doc_track[doc_name]['hnsw_rank'] = rank_1based
        doc_track[doc_name]['hnsw_score'] = score
    else:
        doc_track[doc_name] = {'bm25_rank': 'N/A', 'bm25_score': 0.0, 'hnsw_rank': rank_1based, 'hnsw_score': score}

sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

rows = []
for doc_name, total_score in sorted_rrf:
    t = doc_track[doc_name]
    rows.append({
        'Document': doc_name,
        'BM25 Rank': t['bm25_rank'],
        'BM25 RRF Score': f"1/(60+{t['bm25_rank']})" if t['bm25_rank'] != 'N/A' else '0',
        'HNSW Rank': t['hnsw_rank'],
        'HNSW RRF Score': f"1/(60+{t['hnsw_rank']})" if t['hnsw_rank'] != 'N/A' else '0',
        'Total RRF Score': f"{total_score:.6f}"
    })

df_rrf = pd.DataFrame(rows)
print("=== RRF RANK FUSION MATRIX ===")
print(df_rrf.to_string(index=False))

candidates = [doc_name for doc_name, _ in sorted_rrf]
print(f"\nOUTPUT STEP 4.4 (Candidates): {candidates}")

=== RRF RANK FUSION MATRIX ===
Document  BM25 Rank BM25 RRF Score  HNSW Rank HNSW RRF Score Total RRF Score
    doc4          1       1/(60+1)          1       1/(60+1)        0.032787
    doc3          2       1/(60+2)          2       1/(60+2)        0.032258
    doc1          3       1/(60+3)          3       1/(60+3)        0.031746

OUTPUT STEP 4.4 (Candidates): ['doc4', 'doc3', 'doc1']


### Step 4.5: Reranking using Cross-Encoder
*   **Input:** Original `query` and the `candidates` list (output from Step 4.4).
*   **Action:** Create query-document pairs and feed them into the Cross-Encoder model to calculate deep correlation scores.
*   **Output:** The system's final ranking.

In [9]:
pairs = [[query, corpus[cand]] for cand in candidates]
print("=== CROSS-ENCODER INPUT ===")
for i, pair in enumerate(pairs):
    print(f"  - Pair {i+1}: Query = '{pair[0]}' | Doc = '{pair[1][:45]}...'")

ce_scores = cross_encoder.predict(pairs)
reranked_results = sorted(zip(candidates, ce_scores), key=lambda x: x[1], reverse=True)

print("\n=== FINAL OUTPUT (CROSS-ENCODER RANKING) ===")
for rank, (doc_name, score) in enumerate(reranked_results):
    print(f"  Rank {rank+1}: {doc_name} | Score = {score:.4f} | Content: {corpus[doc_name]}")

=== CROSS-ENCODER INPUT ===
  - Pair 1: Query = 'Amazon capital expenditures 2023' | Doc = 'Amazon purchases of property and equipment in...'
  - Pair 2: Query = 'Amazon capital expenditures 2023' | Doc = 'Amazon net sales were $574,785 million in fis...'
  - Pair 3: Query = 'Amazon capital expenditures 2023' | Doc = 'Apple Net sales in fiscal year 2023 were $383...'

=== FINAL OUTPUT (CROSS-ENCODER RANKING) ===
  Rank 1: doc4 | Score = 4.5093 | Content: Amazon purchases of property and equipment in fiscal year 2023 were $52,729 million.
  Rank 2: doc3 | Score = 3.9284 | Content: Amazon net sales were $574,785 million in fiscal year 2023.
  Rank 3: doc1 | Score = -6.4489 | Content: Apple Net sales in fiscal year 2023 were $383,285 million.


## 5. Temporal Noise Removal Test
We test with the query: **"NVIDIA net income was what in fiscal year 2023?"** to demonstrate the effectiveness of the Year Routing layer.

In [10]:
query_temporal = "NVIDIA net income was what in fiscal year 2023?"
print(f"=== QUERY: '{query_temporal}' ===")

# Step 1: Extract year
match_temp = re.search(r'\b(20\d{2})\b', query_temporal)
year_temp = int(match_temp.group(1)) if match_temp else None
print(f"  - Extracted year: {year_temp}")

# Step 2: Filter documents by year
active_keys_temp = [k for k, text in corpus.items() if not year_temp or str(year_temp) in text]
print(f"  - Active documents list: {active_keys_temp}")
print("    (doc5 containing 2024 NVIDIA data is completely removed)")

# Step 3: Semantic search on filtered documents
temp_texts = [corpus[k] for k in active_keys_temp]
temp_embeddings = bi_encoder.encode(temp_texts, normalize_embeddings=True)
query_temp_vector = bi_encoder.encode([query_temporal], normalize_embeddings=True)[0]

temp_scores = {}
for i, k in enumerate(active_keys_temp):
    sim = np.dot(query_temp_vector, temp_embeddings[i])
    temp_scores[k] = sim

ranked_temp = sorted(temp_scores.items(), key=lambda x: x[1], reverse=True)
print("\n=== FINAL DENSE SEARCH RESULTS (NO 2024 NOISE) ===")
for rank, (doc_name, score) in enumerate(ranked_temp):
    print(f"  Rank {rank+1}: {doc_name} | Cosine Score = {score:.4f} | Content: {corpus[doc_name]}")

=== QUERY: 'NVIDIA net income was what in fiscal year 2023?' ===
  - Extracted year: 2023
  - Active documents list: ['doc1', 'doc3', 'doc4']
    (doc5 containing 2024 NVIDIA data is completely removed)

=== FINAL DENSE SEARCH RESULTS (NO 2024 NOISE) ===
  Rank 1: doc1 | Cosine Score = 0.6251 | Content: Apple Net sales in fiscal year 2023 were $383,285 million.
  Rank 2: doc3 | Cosine Score = 0.6114 | Content: Amazon net sales were $574,785 million in fiscal year 2023.
  Rank 3: doc4 | Cosine Score = 0.5695 | Content: Amazon purchases of property and equipment in fiscal year 2023 were $52,729 million.
